# Net-Payout-Based Duration

## Step 0: Setup, Imports, Pfade, Session


In [42]:
import re
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path
import time
import random
import statsmodels.api as sm

import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="lseg"
)

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

## Step 1: Universum laden: STOXX 600 (EURO subset)

In [2]:
# --- User input ---
INPUT_XLSX = TABLE_DIR / "stoxx600_membership_matrix_1999_2025_eurohq.xlsx"
SHEET_NAME = 0  # or a sheet name string

df_const = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Robust column detection
ric_col = next(c for c in df_const.columns if "RIC" in c.upper())
name_col = next(c for c in df_const.columns if "COMPANY" in c.upper() or "NAME" in c.upper())
country_col = next(c for c in df_const.columns if "COUNTRY" in c.upper())

df_ric = df_const[[ric_col, name_col, country_col]].copy()
df_ric.columns = ["RIC", "CompanyName", "CountryHQ"]

df_ric = (
    df_ric
    .dropna(subset=["RIC"])
    .drop_duplicates(subset=["RIC"])
    .sort_values("RIC")
    .reset_index(drop=True)
)

ric_list = df_ric["RIC"].dropna().astype(str).unique().tolist()


## Step 2: Alle benötigten LSEG Items ziehen

### LSEG Data Items

All data items are obtained from **LSEG Workspace (Refinitiv)** and satisfy the following criteria:
- annual frequency (fiscal year)
- reported (not estimates)
- consolidated
- not forward-looking

#### 1.1 Market-Based Variable

##### Market Equity (Market Capitalization)
- **LSEG Item:** `TR.MarketCapLocalCurn`
- **Description:** Market capitalization of the firm in local currency.
- **Usage:** Market equity $ ME_{i,t} $.

---

#### 1.2 Accounting-Based Variables

##### Book Equity (Common Equity)
- **LSEG Item:** `TR.F.ComEqTot`
- **Description:** Total common equity attributable to common shareholders.
- **Notes:** Excludes minority interest and hybrid debt.
- **Usage:** Book equity $ BE_{i,t} $.

---

##### Total Assets
- **LSEG Item:** `TR.F.TotAssets`
- **Description:** Total assets reported on the consolidated balance sheet.
- **Usage:** Asset base $ A_{i,t} $.

---

##### Revenue (Sales)
- **LSEG Item:** `TR.F.TotRevenue`
- **Description:** Revenue from business activities, total consolidated revenue.
- **Usage:** Firm sales $ Sales_{i,t} $.

---

##### Net Income After Tax
- **LSEG Item:** `TR.F.NetIncAfterTax`
- **Description:** Net income after tax attributable to shareholders.
- **Usage:** Earnings $ NI_{i,t} $.

---

##### Gross Profit (Gross Income)
- **LSEG Item:** `TR.GrossIncomeActValue`
- **Description:** Gross income defined as total revenue minus cost of goods sold.
- **Notes:** Reported, consolidated, all-industry definition.
- **Usage:** Gross profit $ GP_{i,t} $.

---

##### Total Debt
- **LSEG Item:** `TR.F.DebtTot`
- **Description:** Total interest-bearing debt (short-term and long-term).
- **Usage:** Total debt $ Debt_{i,t} $.

---

#### 1.3 Payout Variables

##### Cash Dividends Paid
- **LSEG Item:** `TR.F.DivPaidCashTotCF`
- **Description:** Total cash dividends paid to shareholders during the fiscal year.
- **Usage:** Cash dividends component of payouts $ Div_{i,t} $.

---

##### Share Repurchases (Net)
- **LSEG Item:** `TR.F.ComStockBuybackNet`
- **Description:** Net cash outflow from common share repurchases, net of equity issuance.
- **Notes:** Positive values indicate net repurchases; negative values indicate net issuance.
- **Usage:** Share repurchase component of payouts $ Buyback_{i,t} $.

---

##### Total Payouts (Constructed)
- **Construction:**
  $$
  PO_{i,t} = Div_{i,t} + Buyback_{i,t}
  $$
- **Description:** Total cash distributed to shareholders via dividends and net share repurchases.
- **Usage:** Total payout $ PO_{i,t} $, used in the valuation identity and duration calculations.

In [26]:
# ============================================================
# STEP 2 (v3, FINAL): Annual firm-year pull for paper-style states
# - Robust to missing/garbled Date (1970 issue)
# - Uses the returned time index if it's a real datetime (typical for FY series)
# - If the time index is numeric (0,1,2...), it does NOT convert it to datetime
#   (avoids 1970 epoch bug) and instead uses an internally generated year grid.
# - Returns ONE row per (RIC, year) by keeping the last observation in that year.
#
# Output columns:
#   RIC | year | date_end |
#   ME | BE | assets | sales | net_income | gross_profit | debt | dividends | buybacks
# ============================================================

def pull_step2_fundamentals_v3(
    rics,
    start="1999-01-01",
    end="2026-01-01",
    batch_size=50,
    frq="Y",
):
    fields = [
        "TR.MarketCapLocalCurn",
        "TR.F.ComEqTot",
        "TR.F.TotAssets",
        "TR.F.TotRevenue",
        "TR.F.NetIncAfterTax",
        "TR.GrossIncomeActValue",
        "TR.F.DebtTot",
        "TR.F.DivPaidCashTotCF",
        "TR.F.ComStockBuybackNet",
    ]

    # Clean universe
    rics = list(pd.Series(rics).dropna().astype(str).unique())

    def _chunks(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i : i + n]

    # Map LSEG "title" columns to canonical names (your titles match this list)
    title_map = {
        "Company Market Cap": "ME",
        "Common Equity - Total": "BE",
        "Total Assets": "assets",
        "Revenue from Business Activities - Total": "sales",
        "Net Income after Tax": "net_income",
        "Gross Income - Actual": "gross_profit",
        "Debt - Total": "debt",
        "Dividends Paid - Cash - Total - Cash Flow": "dividends",
        "Common Stock Buyback - Net": "buybacks",
    }

    # Year grid implied by request window (FY series should align to these years)
    start_year = pd.to_datetime(start).year
    end_year = pd.to_datetime(end).year
    # If you use end="2026-01-01", you want last full year = 2025
    if pd.to_datetime(end).month == 1 and pd.to_datetime(end).day == 1:
        end_year = end_year - 1
    requested_years = list(range(start_year, end_year + 1))

    out = []
    for batch in _chunks(rics, batch_size):
        df = ld.get_data(
            universe=batch,
            fields=fields,
            parameters={"SDate": start, "EDate": end, "FRQ": frq},
        )
        df = df.rename(columns=lambda c: str(c).strip())

        if "Instrument" not in df.columns:
            raise ValueError(f"Expected 'Instrument' column, got: {list(df.columns)}")

        # --------------------------------------------------------------------
        # TIME HANDLING (fix the 1970 bug)
        # LSEG often returns the time axis as the dataframe index.
        # - If index is datetime-like -> use it
        # - If index is numeric (0..n) -> DO NOT pd.to_datetime it; build years ourselves
        # --------------------------------------------------------------------
        idx = df.index
        has_datetime_index = pd.api.types.is_datetime64_any_dtype(idx) or isinstance(
            idx[0], (pd.Timestamp,)
        ) if len(idx) else False

        if has_datetime_index:
            df = df.reset_index().rename(columns={df.index.name or "index": "date_end"})
            df["date_end"] = pd.to_datetime(df["date_end"], errors="coerce")
            df = df.dropna(subset=["date_end"]).copy()
            df["year"] = df["date_end"].dt.year.astype(int)
        else:
            # No reliable dates: assign years by position within each RIC's time series.
            # LSEG returns one row per period per instrument; with FRQ="Y"
            # we expect len(requested_years) rows per instrument in a long panel.
            df = df.reset_index(drop=False)

            # Identify the time index column name after reset_index
            # (it will be "index" or the original index name)
            time_col = "index" if "index" in df.columns else df.columns[0]
            # We keep it but do not interpret it as date.
            df = df.rename(columns={"Instrument": "RIC"})

            # Rename value columns now (still safe)
            df = df.rename(columns={c: title_map[c] for c in df.columns if c in title_map})

            # Build year by order within each RIC
            df["_t"] = df.groupby("RIC").cumcount()

            # Map _t -> requested_years. If returned series is shorter, we truncate.
            # If longer (rare), we clip at last year.
            df["year"] = df["_t"].apply(
                lambda k: requested_years[k] if k < len(requested_years) else requested_years[-1]
            ).astype(int)

            # Create a synthetic year-end date (useful later, but not relied upon)
            df["date_end"] = pd.to_datetime(df["year"].astype(str) + "-12-31")

            df = df.drop(columns=["_t"])

        # Standardize RIC col (for datetime-index branch we still need to rename)
        if "RIC" not in df.columns:
            df = df.rename(columns={"Instrument": "RIC"})

        # Rename titles -> canonical names (for datetime-index branch)
        df = df.rename(columns={c: title_map[c] for c in df.columns if c in title_map})

        out.append(df)

    panel = pd.concat(out, ignore_index=True)

    # Keep only canonical columns
    keep = [
        "RIC", "year", "date_end",
        "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt",
        "dividends", "buybacks",
    ]
    keep = [c for c in keep if c in panel.columns]
    panel = panel[keep].copy()

    # One row per (RIC, year): keep the last observation (if duplicates exist)
    panel = (
        panel.sort_values(["RIC", "year", "date_end"])
             .groupby(["RIC", "year"], as_index=False)
             .last()
             .sort_values(["RIC", "year"])
             .reset_index(drop=True)
    )

    return panel

In [57]:
ld.open_session()

panel_step2 = pull_step2_fundamentals_v3(
    ric_list,
    start="1990-01-01",
    end="2026-01-01",
    batch_size=50,
    frq="Y",
)

ld.close_session()

panel_step2.head()
panel_step2[["year"]].describe()

,year
count,23522.000000
mean,2007.489584
std,10.394191
min,1990.000000
25%,1998.000000
50%,2007.000000
75%,2016.000000
max,2025.000000


In [58]:
# 1) Year range should match 1999..2025 (given end=2026-01-01)
panel_step2["year"].min(), panel_step2["year"].max()

# 2) Missingness (buybacks can be NA; we handle later in Step 3)
panel_step2.isna().mean().sort_values()

# 3) Sample rows for one firm
panel_step2[panel_step2["RIC"] == "SAPG.DE"]

,RIC,year,date_end,ME,BE,assets,sales,net_income,gross_profit,debt,dividends,buybacks
18585,SAPG.DE,1990,1990-12-31,<NA>,None,None,None,None,<NA>,None,None,None
18586,SAPG.DE,1991,1991-12-31,<NA>,None,None,None,None,<NA>,None,None,None
18587,SAPG.DE,1992,1992-12-31,<NA>,None,None,None,None,<NA>,None,None,None
18588,SAPG.DE,1993,1993-12-31,<NA>,259568060.61877,341144680.26362,361536023.06949,63031551.82199,<NA>,511291.8812,None,None
18589,SAPG.DE,1994,1994-12-31,<NA>,458755106.52766,552235623.75053,424974563.22891,65057290.25529,<NA>,511291.8812,None,None
18590,SAPG.DE,1995,1995-12-31,<NA>,515493166.58401,667841785.84028,563307649.43783,56449180.14347,<NA>,511291.8812,19224574.73298,None
18591,SAPG.DE,1996,1996-12-31,<NA>,630920887.80722,894622231.99358,936248549.20929,233831161.19499,<NA>,10233507.00214,24062929.80474,None
18592,SAPG.DE,1997,1997-12-31,<NA>,780508019.61316,1134128733.06985,1406329793.48921,339238584.13052,<NA>,10232995.71026,45464585.36785,None
18593,SAPG.DE,1998,1998-12-31,<NA>,1128456972.23174,1721572938.34331,1941284774.23907,494886058.60428,<NA>,50667491.5509,68316264.70603,None
18594,SAPG.DE,1999,1999-12-31,<NA>,1558325621.34746,2592382773.55394,3118831391.27634,852290843.2737,<NA>,86029971.93008,122808730.82016,None


## Step 3:  Firm-year Masterpanel (ME, BE, A, Sales, NI, GP, Debt, Div, Buyback)

In [61]:
# ============================================================
# STEP 3: Build firm-year masterpanel (RIC × year)
# - enforce one row per (RIC, year)
# - handle missing payout components (set to 0)
# - construct payouts + useful lagged inputs for state variables
# ============================================================

def build_masterpanel_firmyear(panel_step2):
    """
    Input: panel_step2 from Step 2 (v3)
      expected cols:
        RIC, year, (optional) date_end,
        ME, BE, assets, sales, net_income, gross_profit, debt,
        dividends, buybacks

    Output: masterpanel (RIC, year) with:
      - cleaned types
      - payouts: PO = dividends + buybacks
      - lags: *_lag1
      - deltas/averages: dBE, avgBE, avgAssets
    """
    df = panel_step2.copy()

    # ---- basic checks / types ----
    if "RIC" not in df.columns or "year" not in df.columns:
        raise ValueError("panel_step2 must contain columns: RIC, year")

    df["RIC"] = df["RIC"].astype(str)
    df["year"] = df["year"].astype(int)

    # ---- keep only relevant columns if they exist ----
    cols = [
        "RIC", "year", "date_end",
        "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt",
        "dividends", "buybacks"
    ]
    cols = [c for c in cols if c in df.columns]
    df = df[cols].copy()

    # ---- enforce numeric on value columns ----
    value_cols = [c for c in ["ME","BE","assets","sales","net_income","gross_profit","debt","dividends","buybacks"] if c in df.columns]
    for c in value_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # ---- deduplicate: one row per (RIC, year) ----
    # (if duplicates exist, keep the last by date_end; if no date_end, keep last row)
    sort_cols = ["RIC", "year"] + (["date_end"] if "date_end" in df.columns else [])
    df = df.sort_values(sort_cols).groupby(["RIC", "year"], as_index=False).last()

    # ---- payout components: treat missing as zero (paper-style) ----
    # Rationale: Many firms simply have no buybacks reported; LSEG often stores as NA.
    if "dividends" in df.columns:
        df["dividends"] = df["dividends"].fillna(0.0)
    else:
        df["dividends"] = 0.0

    if "buybacks" in df.columns:
        df["buybacks"] = df["buybacks"].fillna(0.0)
    else:
        df["buybacks"] = 0.0

    # ---- total payouts ----
    df["PO"] = df["dividends"] + df["buybacks"]

    # ---- sort + lag variables (t-1) ----
    df = df.sort_values(["RIC", "year"]).reset_index(drop=True)

    lag_vars = [c for c in ["ME","BE","assets","sales","net_income","gross_profit","debt","dividends","buybacks","PO"] if c in df.columns]
    for c in lag_vars:
        df[f"{c}_lag1"] = df.groupby("RIC")[c].shift(1)

    # ---- deltas and averages used in profitability states ----
    # ΔBE_t = BE_t - BE_{t-1}
    df["dBE"] = df["BE"] - df["BE_lag1"]

    # avg(BE) = 0.5*(BE_t + BE_{t-1})
    df["avgBE"] = 0.5 * (df["BE"] + df["BE_lag1"])

    # avg(Assets) = 0.5*(A_t + A_{t-1})
    df["avgAssets"] = 0.5 * (df["assets"] + df["assets_lag1"])

    # ---- basic sanity: market cap should be > 0 for valuation states ----
    # (We don't drop rows here; just mark as missing where unusable)
    df.loc[df["ME"] <= 0, "ME"] = pd.NA
    df.loc[df["BE"] <= 0, "BE"] = pd.NA
    df.loc[df["assets"] <= 0, "assets"] = pd.NA
    df.loc[df["sales"] <= 0, "sales"] = pd.NA
    # debt can be zero; keep it (but negative debt usually indicates bad data)
    df.loc[df["debt"] < 0, "debt"] = pd.NA

    return df

In [62]:
master = build_masterpanel_firmyear(panel_step2)

master.head()

,RIC,year,date_end,ME,BE,assets,sales,net_income,gross_profit,debt,...,sales_lag1,net_income_lag1,gross_profit_lag1,debt_lag1,dividends_lag1,buybacks_lag1,PO_lag1,dBE,avgBE,avgAssets
0,1COVG.DE^L25,1990,1990-12-31,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,...,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1COVG.DE^L25,1991,1991-12-31,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,...,NaN,NaN,<NA>,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,1COVG.DE^L25,1992,1992-12-31,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,...,NaN,NaN,<NA>,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,1COVG.DE^L25,1993,1993-12-31,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,...,NaN,NaN,<NA>,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,1COVG.DE^L25,1994,1994-12-31,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,...,NaN,NaN,<NA>,NaN,0.0,0.0,0.0,NaN,NaN,NaN


## Step 4: Construction of Firm-Level State Variables


In this step, we construct the **firm-level state variables** that summarize valuation, growth, profitability, and capital structure. These variables form the **state vector** used later in the VAR and valuation identity, following the methodology of the paper as closely as possible.

All state variables are:
- constructed at **annual (firm-year) frequency**
- based on **reported, consolidated accounting data**
- computed using **information available at time $ t $** (with lags where required)
- transformed using logs or log(1+x), exactly as in the paper

Observations for which the underlying transformation is not well-defined (e.g. non-positive denominators) are set to missing (`NaN`) rather than forced or winsorized.

---

### 4.1 Valuation States

These variables capture how the firm is priced relative to fundamentals.

- **Book-to-Market**
  $$
  bm_{i,t} = \log\left(\frac{BE_{i,t}}{ME_{i,t}}\right)
  $$

- **Payout Yield**
  $$
  py_{i,t} = \log\left(1 + \frac{PO_{i,t}}{ME_{i,t}}\right)
  $$

- **Sales Yield**
  $$
  sy_{i,t} = \log\left(\frac{Sales_{i,t}}{ME_{i,t}}\right)
  $$

---

### 4.2 Growth States

These variables capture firm-level growth in size and operations.

- **Book Equity Growth**
  $$
  beg_{i,t} = \log\left(\frac{BE_{i,t}}{BE_{i,t-1}}\right)
  $$

- **Asset Growth**
  $$
  ag_{i,t} = \log\left(\frac{Assets_{i,t}}{Assets_{i,t-1}}\right)
  $$

- **Sales Growth**
  $$
  sg_{i,t} = \log\left(\frac{Sales_{i,t}}{Sales_{i,t-1}}\right)
  $$

---

### 4.3 Profitability States

These variables measure different notions of firm profitability.

- **Clean-Surplus Profitability**
  $$
  csprof_{i,t} =
  \log\left(
    1 + \frac{NI_{i,t} - \Delta BE_{i,t}}{BE_{i,t-1}}
  \right)
  $$

- **Return on Equity**
  $$
  roe_{i,t} =
  \log\left(
    1 + \frac{NI_{i,t}}{\frac{1}{2}(BE_{i,t} + BE_{i,t-1})}
  \right)
  $$

- **Gross Profitability**
  $$
  gp_{i,t} =
  \log\left(
    1 + \frac{GrossProfit_{i,t}}{\frac{1}{2}(Assets_{i,t} + Assets_{i,t-1})}
  \right)
  $$

---

### 4.4 Capital Structure State

- **Market Leverage**
  $$
  lev_{i,t} = \frac{Debt_{i,t}}{Debt_{i,t} + ME_{i,t}}
  $$

---

### Remarks

- Log transformations are only applied where economically meaningful (strictly positive inputs).
- Missing values arise naturally from lag construction or accounting edge cases and are retained.
- No standardization, demeaning, or winsorization is applied at this stage.
- This step completes the data preparation required for estimating the VAR in Step 5.

After Step 4, the dataset consists of a firm-year panel with a complete, paper-consistent state vector for each firm.

In [66]:
def build_state_variables(master):
    df = master.copy()

    def safe_log(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index)
        m = x > 0
        out.loc[m] = np.log(x.loc[m])
        return out

    def safe_log1p(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index)
        m = x > -1  # log(1+x) defined iff 1+x>0
        out.loc[m] = np.log1p(x.loc[m])
        return out

    # Valuation
    df["bm"] = safe_log(df["BE"] / df["ME"])
    df["py"] = safe_log1p(df["PO"] / df["ME"])
    df["sy"] = safe_log(df["sales"] / df["ME"])

    # Growth
    df["beg"] = safe_log(df["BE"] / df["BE_lag1"])
    df["ag"]  = safe_log(df["assets"] / df["assets_lag1"])
    df["sg"]  = safe_log(df["sales"] / df["sales_lag1"])

    # Profitability
    df["csprof"] = safe_log1p((df["net_income"] - df["dBE"]) / df["BE_lag1"])
    df["roe"]    = safe_log1p(df["net_income"] / df["avgBE"])
    df["gp"]     = safe_log1p(df["gross_profit"] / df["avgAssets"])

    # Capital structure (robust to pd.NA)
    debt = pd.to_numeric(df["debt"], errors="coerce").to_numpy(dtype=float)
    me   = pd.to_numeric(df["ME"],   errors="coerce").to_numpy(dtype=float)
    denom = debt + me

    lev = np.full(len(df), np.nan, dtype=float)
    mask = denom > 0
    lev[mask] = debt[mask] / denom[mask]
    df["lev"] = lev

    # Clean
    state_vars = ["bm","py","sy","beg","ag","sg","csprof","roe","gp","lev"]
    df[state_vars] = df[state_vars].replace([np.inf, -np.inf], np.nan)

    return df

In [64]:
state_panel = build_state_variables(master)

In [65]:
checks = pd.DataFrame({
    "ME<=0": (master["ME"] <= 0),
    "BE<=0": (master["BE"] <= 0),
    "sales<=0": (master["sales"] <= 0),
    "BE_lag1<=0": (master["BE_lag1"] <= 0),
    "assets_lag1<=0": (master["assets_lag1"] <= 0),
    "sales_lag1<=0": (master["sales_lag1"] <= 0),
    "avgBE<=0": (master["avgBE"] <= 0),
    "avgAssets<=0": (master["avgAssets"] <= 0),
    "1+PO/ME<=0": (1 + master["PO"]/master["ME"] <= 0),
})
checks.mean().sort_values(ascending=False)

BE_lag1<=0        0.013689
avgBE<=0          0.012286
1+PO/ME<=0        0.005413
sales_lag1<=0      0.00136
assets_lag1<=0    0.000765
avgAssets<=0      0.000723
ME<=0                  0.0
BE<=0                  0.0
sales<=0               0.0
dtype: Float64

## Step 5: Estimation of the Firm-Level State VAR

In this step, we estimate the **dynamic law of motion** for the firm-level state variables constructed in Step 4. The objective is to characterize how valuation, growth, profitability, and capital structure jointly evolve over time and to obtain **long-horizon expectations** required for valuation and duration calculations.

Following the paper, the state dynamics are modeled using a **pooled first-order vector autoregression (VAR(1))**:

$$
X_{i,t+1} = \mu + \Phi X_{i,t} + \varepsilon_{i,t+1},
$$

where $ X_{i,t} $ denotes the vector of state variables for firm $ i $ at time $ t $, $ \mu $ is a vector of intercepts, $ \Phi $ is the transition matrix, and $ \varepsilon_{i,t+1} $ is an innovation term.

---

### State Vector Specification

The baseline VAR includes a compact set of core state variables that balance parsimony with economic content:

- **Book-to-market ($ bm $)** — valuation anchor  
- **Payout yield ($ py $)** — payout policy  
- **Sales yield ($ sy $)** — scale-adjusted valuation  
- **Asset growth ($ ag $)** — investment dynamics  
- **Return on equity ($ roe $)** — profitability  
- **Market leverage ($ lev $)** — capital structure  

These variables jointly capture the main channels through which firm characteristics affect expected cash flows and discount rates.

---

### Estimation Strategy

The VAR is estimated on an **unbalanced firm-year panel** using pooled ordinary least squares. To ensure reliable estimation:

- Observations must have valid state values at both $ t $ and $ t+1 $.
- Firms are required to have a minimum time-series length to enter the estimation sample.
- All state variables are **demeaned and standardized** using pooled moments prior to estimation.

This approach improves numerical stability and ensures comparability across state variables with different units and scales.

---

### Role in the Overall Framework

The estimated transition matrix $ \Phi $ summarizes how shocks to firm-level states propagate over time. These dynamics are a key input into:

- the computation of expected future states,
- the mapping from states to expected payout-to-book ratios,
- the valuation identity used to infer firm-specific discount rates,
- and ultimately, the calculation of equity duration.

With the estimation of the VAR completed, the framework is ready to move from reduced-form dynamics to structural valuation in Step 6.

In [67]:
# ============================================================
# STEP 5 (PAPER-CLEAN): Extended pooled VAR(1) + forecasting utilities
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ----------------------------
# 5.1 State vector (extended, paper-consistent)
# ----------------------------
# Include beg and csprof because they enter the cashflow identity in Step 6.
var_states = ["bm", "py", "sy", "ag", "roe", "lev", "beg", "csprof"]

# ----------------------------
# 5.2 Build VAR sample (t -> t+1) with strict validity
# ----------------------------
df_var = state_panel.sort_values(["RIC", "year"]).copy()

# create leads
for v in var_states:
    df_var[f"{v}_lead"] = df_var.groupby("RIC")[v].shift(-1)

# keep only rows where ALL states exist at t and t+1
req_cols = var_states + [f"{v}_lead" for v in var_states]
df_var = df_var.dropna(subset=req_cols).copy()

# minimum time-series length per firm (strict, paper style)
min_T = 10
firm_counts = df_var.groupby("RIC").size()
valid_firms = firm_counts[firm_counts >= min_T].index
df_var = df_var[df_var["RIC"].isin(valid_firms)].copy()

print("STEP 5 VAR sample:")
print(f"  firms: {df_var['RIC'].nunique()}")
print(f"  obs  : {len(df_var)}")

# ----------------------------
# 5.3 Pooled standardization (on X_t)
# ----------------------------
Z = df_var[var_states].astype(float)
Z_lead = df_var[[f"{v}_lead" for v in var_states]].astype(float)

mu = Z.mean()
sigma = Z.std().replace(0, np.nan)

Zs = (Z - mu) / sigma
Zs_lead = (Z_lead - mu.values) / sigma.values

# ----------------------------
# 5.4 Estimate pooled VAR(1): Z_{t+1} = c + Phi Z_t + u_{t+1}
# ----------------------------
Y = Zs_lead.to_numpy(dtype=float)                # (N x k)
X = sm.add_constant(Zs.to_numpy(dtype=float))    # (N x (k+1))

var_model = sm.OLS(Y, X).fit()

Phi = var_model.params[1:, :].T                 # (k x k)
const = var_model.params[0, :]                  # (k,)

phi_df = pd.DataFrame(Phi, index=var_states, columns=var_states)

# ----------------------------
# 5.5 Stability diagnostics
# ----------------------------
eigvals = np.linalg.eigvals(Phi)
max_eig = float(np.max(np.abs(eigvals)))

print("\nExtended VAR(1) estimated")
print(f"  Max |eigenvalue|: {max_eig:.4f}")

# ----------------------------
# 5.6 Forecast utilities for Step 6
# ----------------------------
I = np.eye(len(var_states))

# steady state of standardized VAR: zbar = (I - Phi)^{-1} const
zbar = np.linalg.solve(I - Phi, const)

def to_standardized_state(row):
    """Convert a row with raw state variables into standardized z_t."""
    x = row[var_states].to_numpy(dtype=float)
    return (x - mu.to_numpy(dtype=float)) / sigma.to_numpy(dtype=float)

def forecast_states(z0, H):
    """
    Produce E_t[z_{t+h}] for h=1..H in standardized units.
    Returns array shape (H, k).
    """
    out = np.zeros((H, len(var_states)), dtype=float)
    z = z0.copy()
    for h in range(1, H + 1):
        z = const + Phi @ z
        out[h-1, :] = z
    return out

def forecast_states_closedform(z0, H):
    """
    Closed-form version using Phi^h and steady state.
    out[h-1] = Phi^h z0 + (I - Phi^h) zbar
    """
    out = np.zeros((H, len(var_states)), dtype=float)
    for h in range(1, H + 1):
        Phi_h = np.linalg.matrix_power(Phi, h)
        out[h-1, :] = Phi_h @ z0 + (I - Phi_h) @ zbar
    return out

# Optional: a quick view of Phi
phi_df

STEP 5 VAR sample:
  firms: 428
  obs  : 9242

Extended VAR(1) estimated
  Max |eigenvalue|: 0.9528


,bm,py,sy,ag,roe,lev,beg,csprof
bm,0.919722,-0.024997,-0.015335,0.011493,-0.007259,0.009409,0.018750,0.003281
py,0.100607,0.461204,0.032983,-0.005298,0.050094,-0.029635,0.054381,-0.011709
sy,-0.018768,-0.017949,0.938000,0.023822,-0.007020,-0.005075,-0.001478,-0.016075
ag,-0.056390,-0.003144,-0.075547,0.077610,0.032117,-0.091161,0.113631,0.063358
roe,-0.245853,0.005102,0.041050,-0.001653,0.542436,-0.010017,-0.081263,-0.005545
lev,0.015641,-0.004339,-0.024956,0.032512,0.004345,0.925199,-0.001540,-0.002890
beg,-0.247912,-0.048513,-0.007495,0.039425,0.025249,0.029060,0.096682,0.042134
csprof,0.033045,0.022445,0.031291,-0.034572,0.306641,-0.038276,-0.044131,0.022638


In [68]:
max_eig
phi_df.round(3)

,bm,py,sy,ag,roe,lev,beg,csprof
bm,0.920,-0.025,-0.015,0.011,-0.007,0.009,0.019,0.003
py,0.101,0.461,0.033,-0.005,0.050,-0.030,0.054,-0.012
sy,-0.019,-0.018,0.938,0.024,-0.007,-0.005,-0.001,-0.016
ag,-0.056,-0.003,-0.076,0.078,0.032,-0.091,0.114,0.063
roe,-0.246,0.005,0.041,-0.002,0.542,-0.010,-0.081,-0.006
lev,0.016,-0.004,-0.025,0.033,0.004,0.925,-0.002,-0.003
beg,-0.248,-0.049,-0.007,0.039,0.025,0.029,0.097,0.042
csprof,0.033,0.022,0.031,-0.035,0.307,-0.038,-0.044,0.023


### Step 5 Results: VAR(1) Estimation — Interpretation

This section summarizes and interprets the estimated **pooled VAR(1)** transition matrix $ \Phi $ used to model the joint dynamics of firm-level state variables.

### Summary Statistics
- **Number of firms:** 472  
- **Total firm-year observations:** 9,868  
- **Maximum absolute eigenvalue:** 0.947  

The eigenvalue condition $ \max |\lambda(\Phi)| < 1 $ confirms that the VAR is **stable and stationary**, as required for well-defined long-horizon expectations and valuation calculations.

---

### Persistence (Diagonal Elements)

The diagonal entries of $ \Phi $ capture the persistence of each state variable:

- **Book-to-market (bm):** High persistence (~0.88)  
  → Valuation ratios evolve slowly over time, consistent with the asset-pricing literature.

- **Sales yield (sy):** Very high persistence (~0.93)  
  → Reflects strong persistence in scale-adjusted sales and valuation anchors.

- **Leverage (lev):** Very high persistence (~0.93)  
  → Capital structure adjusts gradually, in line with standard corporate finance evidence.

- **Return on equity (roe):** Moderate persistence (~0.47)  
  → Profitability is mean-reverting, as expected.

- **Payout yield (py):** Moderate persistence (~0.32)  
  → Payout policies vary more over time than valuation ratios.

- **Asset growth (ag):** Low persistence (~0.04)  
  → Growth rates are largely transitory and close to white noise.

Overall, the persistence pattern is economically plausible and closely matches what is typically found in firm-level VARs.

---

### Cross-Dynamics (Off-Diagonal Elements)

Several economically intuitive cross-effects emerge:

- **Value firms (high bm)** tend to:
  - have higher future payout yields,
  - exhibit lower profitability,
  - grow more slowly.

- **Higher profitability (roe)** predicts higher subsequent asset growth.

- Most other cross-effects are small in magnitude, which is expected after standardization and indicates that no single variable dominates the system.

---

### Assessment

- The VAR captures **realistic firm dynamics** across valuation, growth, profitability, and leverage.
- The system is **highly persistent but stationary**, which is essential for computing long-horizon expectations.
- No coefficient patterns suggest misspecification or instability.

---

### Implications for Next Steps

With a stable and well-behaved transition matrix $ \Phi $, the model is ready for:

1. Computing expected future state paths $ E_t[X_{t+h}] $,
2. Mapping these expectations into future payout-to-book ratios via accounting identities,
3. Solving for the firm-specific discount rate from the valuation identity,
4. Computing equity duration.

This concludes **Step 5**. The analysis can now proceed to **Step 6: Valuation Identity and Equity Duration**.

## Step 6:  Valuation Identity, Discount Rate, and Equity Duration

### Step 6: Endogenous Discount Rates and Equity Duration

This step constructs firm-year specific **equity discount rates** and **equity duration** using a valuation-identity approach. The procedure follows the present-value framework in which the market-to-book ratio equals the discounted value of expected future payouts to equityholders.

---

#### 6.1 Valuation Identity

For each firm $ i $ and year $ t $, the market value of equity relative to book equity satisfies the identity

$$
\frac{ME_{i,t}}{BE_{i,t}} 
= \sum_{h=1}^{\infty} 
\mathbb{E}_t \left[
\frac{PO_{i,t+h}}{BE_{i,t}}
\right] 
\exp(- r_{i,t} \, h),
$$

where  
- $ ME_{i,t} $ denotes market equity,  
- $ BE_{i,t} $ denotes book equity,  
- $ PO_{i,t+h} $ denotes total payouts to equityholders, and  
- $ r_{i,t} $ is the firm-year specific discount rate.

The discount rate $ r_{i,t} $ is **endogenously determined** such that the valuation identity holds exactly.

---

#### 6.2 Forecasting Expected Payouts

Expected future payouts are constructed from the firm’s forecasted state variables obtained in Step 5. In particular, two key state variables govern the payout process:

- **Book equity growth** $ beg_{i,t} $,
- **Clean-surplus profitability** $ csprof_{i,t} $.

The payout-to-book ratio at horizon $ h $ is defined as

$$
\frac{PO_{i,t+h}}{BE_{i,t+h-1}}
= \max\left\{ 0,\; \exp(csprof_{i,t+h}) - 1 \right\},
$$

which ensures that payouts represent **non-negative distributions to equityholders**, while negative implied values are interpreted as net equity issuance.

Expected payouts relative to initial book equity are then obtained via

$$
\frac{PO_{i,t+h}}{BE_{i,t}}
=
\frac{PO_{i,t+h}}{BE_{i,t+h-1}}
\cdot
\exp\!\left( \sum_{j=1}^{h-1} beg_{i,t+j} \right).
$$

---

#### 6.3 Finite-Horizon Approximation and Terminal Value

The infinite sum in the valuation identity is approximated using a **finite horizon $ H $** combined with a conservative terminal value:

$$
\frac{ME_{i,t}}{BE_{i,t}}
=
\sum_{h=1}^{H}
\mathbb{E}_t \left[
\frac{PO_{i,t+h}}{BE_{i,t}}
\right]
\exp(- r_{i,t} h)
+
TV_{i,t}(r_{i,t}).
$$

The terminal value is specified as a **level perpetuity without growth**, based on the long-run expected payout ratio,

$$
TV_{i,t}(r)
=
\frac{\overline{po}}{\exp(r) - 1}
\cdot
\exp\!\left(
\sum_{j=1}^{H} beg_{i,t+j}
\right)
\exp(- r H),
$$

where $ \overline{po} = \max\{0, \exp(csprof_{\infty}) - 1\} $ is the steady-state payout-to-book ratio implied by the VAR.

This specification avoids explosive terminal values and yields numerically stable discount-rate solutions.

---

#### 6.4 Solving for the Discount Rate

For each firm-year observation, the discount rate $ r_{i,t} $ is obtained as the unique solution to

$$
f(r) =
\sum_{h=1}^{H}
\mathbb{E}_t \left[
\frac{PO_{i,t+h}}{BE_{i,t}}
\right]
\exp(- r h)
+
TV_{i,t}(r)
-
\frac{ME_{i,t}}{BE_{i,t}}
= 0.
$$

The root is computed using a robust bracketing method, and observations for which no valid solution exists are discarded.

---

#### 6.5 Equity Duration

Equity duration measures the weighted average maturity of expected payouts and is defined as

$$
Dur_{i,t}
=
\frac{1}{ME_{i,t}/BE_{i,t}}
\left[
\sum_{h=1}^{H}
h \cdot
\mathbb{E}_t \left[
\frac{PO_{i,t+h}}{BE_{i,t}}
\right]
\exp(- r_{i,t} h)
+
Dur^{TV}_{i,t}
\right],
$$

where the terminal contribution equals

$$
Dur^{TV}_{i,t}
=
\left(
H + \frac{1}{\exp(r_{i,t}) - 1}
\right)
TV_{i,t}(r_{i,t}).
$$

This measure captures the **effective maturity of equity cash flows**, analogous to duration for fixed-income securities.

---

#### 6.6 Sample Construction and Diagnostics

Equity duration and discount rates are computed for firm-year observations satisfying:

- positive market and book equity,
- valid VAR forecasts for all required state variables,
- a well-defined solution to the valuation identity.

The resulting panel exhibits substantial cross-sectional and time-series variation. Discount rates and equity duration are negatively correlated, consistent with asset pricing theory, and the valuation identity is satisfied up to numerical precision.

---

In [87]:
# ============================================================
# STEP 6 (v3): Stable valuation identity + duration
# Fixes:
#  (A) payouts truncated at 0 (distributions cannot be negative)
#  (B) conservative terminal value (no growth perpetuity)
# ============================================================

import numpy as np
import pandas as pd
from scipy.optimize import brentq

H = 30
DR_HI = 1.50

k = len(var_states)
I = np.eye(k)

# steady state standardized
try:
    zbar
except NameError:
    zbar = np.linalg.solve(I - Phi, const)

mu_arr = mu.to_numpy(dtype=float)
sig_arr = sigma.to_numpy(dtype=float)

def forecast_states_closedform(z0, H):
    out = np.zeros((H, k), dtype=float)
    for h in range(1, H + 1):
        Phi_h = np.linalg.matrix_power(Phi, h)
        out[h-1, :] = Phi_h @ z0 + (I - Phi_h) @ zbar
    return out

def unstandardize(z):
    return mu_arr + sig_arr * z

# indices
ix_beg = var_states.index("beg")
ix_cs  = var_states.index("csprof")

# long-run payout ratio level (per BE_{t-1})
xbar_raw = unstandardize(zbar)
csprof_bar = float(xbar_raw[ix_cs])
po_over_be_lag_bar = max(0.0, np.exp(csprof_bar) - 1.0)

def payout_path_to_BE0(beg_path, csprof_path):
    # distributions: max(0, exp(csprof)-1)
    po_over_be_lag = np.maximum(0.0, np.exp(csprof_path) - 1.0)

    # scale by BE_{t+h-1}/BE_t = exp(sum_{j=1}^{h-1} beg_{t+j})
    cum_beg = np.cumsum(beg_path)
    scale = np.exp(np.r_[0.0, cum_beg[:-1]])
    return po_over_be_lag * scale

def terminal_value_BE0(sum_beg_to_H, dr):
    # conservative perpetuity with no growth beyond H
    # TV at H: po_bar / (e^{dr}-1), discounted and scaled by BE growth up to H
    if dr <= 1e-6:
        return np.nan
    annuity = po_over_be_lag_bar / (np.exp(dr) - 1.0)
    return annuity * np.exp(sum_beg_to_H) * np.exp(-dr * H)

def solve_dr(me_be, payout_path, sum_beg_to_H):
    h = np.arange(1, H + 1, dtype=float)

    def f(dr):
        disc = np.exp(-dr * h)
        pv_finite = float(np.sum(payout_path * disc))
        tv = terminal_value_BE0(sum_beg_to_H, dr)
        if not np.isfinite(tv):
            return np.nan
        return (pv_finite + tv) - me_be

    # bracket: low rate must be > 0
    dr_lo = 1e-4
    f_lo = f(dr_lo)
    f_hi = f(DR_HI)

    if not (np.isfinite(f_lo) and np.isfinite(f_hi)):
        raise ValueError("Non-finite at endpoints.")
    if f_lo * f_hi > 0:
        raise ValueError("No sign change.")
    return float(brentq(f, dr_lo, DR_HI, maxiter=200))

def compute_duration(me_be, payout_path, sum_beg_to_H, dr):
    h = np.arange(1, H + 1, dtype=float)
    disc = np.exp(-dr * h)

    pv_finite = float(np.sum(payout_path * disc))
    dur_num_finite = float(np.sum(h * payout_path * disc))

    tv = terminal_value_BE0(sum_beg_to_H, dr)
    if not np.isfinite(tv):
        return np.nan, np.nan

    # terminal is an annuity starting at H (effective maturity ~ H + 1/(e^{dr}-1) )
    # duration contribution of perpetuity: H + 1/(e^{dr}-1)
    dur_tv = (H + (1.0 / (np.exp(dr) - 1.0))) * tv

    pv_total = pv_finite + tv
    dur = (dur_num_finite + dur_tv) / me_be
    pv_check = pv_total - me_be
    return dur, pv_check

# Run
need_cols = ["RIC", "year", "ME", "BE"] + var_states
df6 = state_panel.dropna(subset=need_cols).copy()
df6 = df6[(df6["ME"] > 0) & (df6["BE"] > 0)].copy()

res = []
for row in df6.itertuples(index=False):
    x0 = np.array([getattr(row, v) for v in var_states], dtype=float)
    z0 = (x0 - mu_arr) / sig_arr

    z_path = forecast_states_closedform(z0, H)
    x_path = unstandardize(z_path)

    beg_path = x_path[:, ix_beg]
    cs_path  = x_path[:, ix_cs]

    payout_path = payout_path_to_BE0(beg_path, cs_path)
    me_be = float(getattr(row, "ME") / getattr(row, "BE"))
    sum_beg_to_H = float(np.sum(beg_path))

    try:
        dr = solve_dr(me_be, payout_path, sum_beg_to_H)
        dur, pv_check = compute_duration(me_be, payout_path, sum_beg_to_H, dr)
        res.append({"RIC": getattr(row, "RIC"), "year": int(getattr(row, "year")),
                    "discount_rate": dr, "equity_duration": dur,
                    "pv_check": pv_check, "status": "ok"})
    except Exception:
        res.append({"RIC": getattr(row, "RIC"), "year": int(getattr(row, "year")),
                    "discount_rate": np.nan, "equity_duration": np.nan,
                    "pv_check": np.nan, "status": "fail"})

duration_panel_v3 = pd.DataFrame(res)

print("STEP 6 v3 done.")
print("  ok:", (duration_panel_v3["status"]=="ok").sum())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","equity_duration"].describe())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","discount_rate"].describe())
print(duration_panel_v3.loc[duration_panel_v3["status"]=="ok","pv_check"].abs().describe())

STEP 6 v3 done.
  ok: 10661
count    10661.000000
mean        32.612680
std         10.601656
min          1.000000
25%         25.806061
50%         30.664804
75%         38.183531
max        224.605487
Name: equity_duration, dtype: float64
count    10661.000000
mean         0.053596
std          0.072350
min          0.004510
25%          0.039838
50%          0.047740
75%          0.056238
max          1.434543
Name: discount_rate, dtype: float64
count    1.066100e+04
mean     3.600251e-11
std      3.180161e-09
min      0.000000e+00
25%      1.776357e-15
50%      5.734302e-14
75%      2.259526e-12
max      3.283071e-07
Name: pv_check, dtype: float64


In [88]:
save_parquet(duration_panel_v3, "NP_duration_panel")

Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/NP_duration_panel.parquet


In [83]:
ok = duration_panel_v3.query("status=='ok'")
(ok["discount_rate"] > 0.5).mean(), ok["discount_rate"].quantile([0.99, 0.995, 0.999])

(np.float64(0.003470593752931245),
 0.990    0.106194
 0.995    0.204390
 0.999    1.434543
 Name: discount_rate, dtype: float64)

In [84]:
ok[["equity_duration","discount_rate"]].corr()

,equity_duration,discount_rate
equity_duration,1.000000,-0.331555
discount_rate,-0.331555,1.000000


In [85]:
ok.groupby("RIC")["equity_duration"].count().describe()

count    535.000000
mean      19.927103
std        8.190402
min        1.000000
25%       14.000000
50%       23.000000
75%       26.000000
max       35.000000
Name: equity_duration, dtype: float64